In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = (SparkSession.builder
         .appName("windows")
         .master("spark://spark-master:7077")
         .config("spark.executor.memory", "512m")
         .getOrCreate())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/24 16:12:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = (spark.read.format("csv")
      .option("header", "true")
      .option("nullValue", "null")
      .option("dateFormat", "LLLL d, y")
      .load("../data/netflix_titles_extended.csv"))


df.show()

+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|            director|                cast|             country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|     Kirsten Johnson|                null|       United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|                null|Ama Qamata, Khosi...|        South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglan

In [4]:
from pyspark.sql.window import *

window_spec = Window.partitionBy("country").orderBy("date_added")

In [5]:
result = df.withColumn("row_number", row_number().over(window_spec))

result.select("title", "country", "date_added", "row_number").show()

+--------------------+-------+------------------+----------+
|               title|country|        date_added|row_number|
+--------------------+-------+------------------+----------+
|            Kikoriki|   null|              null|         1|
|                null|   null|              null|         2|
|The Memphis Belle...|   null|              null|         3|
|     Fit for Fashion|   null| December 14, 2018|         4|
|        Lego Friends|   null|  February 1, 2019|         5|
|Nightmare Tenants...|   null|     July 12, 2019|         6|
|            Satrangi|   null|     April 1, 2017|         7|
|              Buddha|   null|     April 1, 2018|         8|
|          Fishpeople|   null|     April 1, 2018|         9|
|Kicko & Super Speedo|   null|     April 1, 2019|        10|
|  Pokémon the Series|   null|     April 1, 2020|        11|
|    Operation Odessa|   null|     April 1, 2020|        12|
|Glimpses of a Future|   null|     April 1, 2021|        13|
|Seven Souls in th...|  

In [7]:
df = df.withColumn("lead_date_added", lead("date_added").over(window_spec))
df = df.withColumn("lag_date_added", lag("date_added").over(window_spec))

df.select("title", "country", "date_added", "lead_date_added", "lag_date_added").show(3)

+--------------------+-------+----------+------------------+--------------+
|               title|country|date_added|   lead_date_added|lag_date_added|
+--------------------+-------+----------+------------------+--------------+
|            Kikoriki|   null|      null|              null|          null|
|                null|   null|      null|              null|          null|
|The Memphis Belle...|   null|      null| December 14, 2018|          null|
+--------------------+-------+----------+------------------+--------------+
only showing top 3 rows



In [9]:
spark.stop()